In [2]:
import duckdb
import pandas as pd

DB_PATH = "lg_hv.duckdb"  # 파일로 남기고 싶으면, 메모리면 ":memory:" 사용

con = duckdb.connect(DB_PATH)
con.execute("PRAGMA threads=8;")  # 옵션(선택)
print("connected:", DB_PATH)


connected: lg_hv.duckdb


In [10]:
#1) 셀: 원천 parquet 로드 확인
SHA_PAIN_HISTORY_PARQUET = f"../../dataset/processed/sha_pain_history.parquet"

df = con.execute(f"""
SELECT
  COUNT(*) AS n_rows,
  MIN(p_mt) AS min_p_mt,
  MAX(p_mt) AS max_p_mt
FROM read_parquet('{SHA_PAIN_HISTORY_PARQUET}')
""").fetchdf()

df


,n_rows,min_p_mt,max_p_mt
0,22893471,202302,202312


In [11]:
#2) 셀: 월×고객 중복 제거 clean 테이블 생성
con.execute(f"""
DROP TABLE IF EXISTS sha_pain_history_clean;

CREATE TABLE sha_pain_history_clean AS
SELECT
  sha2_hash,
  p_mt,
  STB_RES_1M_YN,
  VOC_TOTAL_MONTH1_YN,
  VOC_STOP_CANCEL_MONTH1_YN,
  cancel_yn
FROM read_parquet('{SHA_PAIN_HISTORY_PARQUET}')
QUALIFY COUNT(*) OVER (PARTITION BY sha2_hash, p_mt) = 1;
""")

# 중복 남았는지 체크 (0이면 정상)
dup = con.execute("""
SELECT sha2_hash, p_mt, COUNT(*) AS cnt
FROM sha_pain_history_clean
GROUP BY 1,2
HAVING COUNT(*) > 1
LIMIT 10
""").fetchdf()

dup


,sha2_hash,p_mt,cnt


In [12]:
# 3) 셀: pain_stage(0~3) 뷰 생성
con.execute("""
CREATE OR REPLACE VIEW v_sha_pain_stage AS
SELECT
  *,
  (CASE WHEN STB_RES_1M_YN = 'Y' THEN 1 ELSE 0 END
 + CASE WHEN VOC_TOTAL_MONTH1_YN = 'Y' THEN 1 ELSE 0 END
 + CASE WHEN VOC_STOP_CANCEL_MONTH1_YN = 'Y' THEN 1 ELSE 0 END) AS pain_stage
FROM sha_pain_history_clean;
""")

con.execute("SELECT * FROM v_sha_pain_stage LIMIT 5").fetchdf()


,sha2_hash,p_mt,STB_RES_1M_YN,VOC_TOTAL_MONTH1_YN,VOC_STOP_CANCEL_MONTH1_YN,cancel_yn,pain_stage
0,22d6712d4c5d6f3f8ca7249138d6545717c6abebf50541...,202307,N,N,N,유지,0
1,22d6cd5cfed231d3ef241ecdf4600b829467bf6b9ba4f8...,202305,N,N,N,유지,0
2,22d76c684eae54adaea7b609f439ea34b093013a6bc66d...,202303,N,N,N,유지,0
3,22d80a136d76633cfaae65267ba459bc29594666cb2654...,202312,N,Y,N,유지,1
4,22d8bfc67fd968c90240b27b2836affd8a91ae4ed30171...,202309,N,N,N,유지,0


In [13]:
#4) 셀: pain_stage 분포 + 해지율
pain_dist = con.execute("""
SELECT
  pain_stage,
  COUNT(*) AS cnt,
  ROUND(AVG(CASE WHEN cancel_yn='해지' THEN 1 ELSE 0 END)*100, 2) AS churn_rate_percent
FROM v_sha_pain_stage
GROUP BY pain_stage
ORDER BY pain_stage;
""").fetchdf()

pain_dist


,pain_stage,cnt,churn_rate_percent
0,0,16694304,3.14
1,1,4498259,5.86
2,2,795402,10.66
3,3,65632,25.09


In [14]:
#5) 셀: 2-signal 조합(0건도 출력) + 해지율
combo = con.execute("""
WITH s AS (
  SELECT
    STB_RES_1M_YN,
    VOC_TOTAL_MONTH1_YN,
    VOC_STOP_CANCEL_MONTH1_YN,
    CASE WHEN cancel_yn='해지' THEN 1 ELSE 0 END AS churn
  FROM sha_pain_history_clean
  WHERE STB_RES_1M_YN IN ('Y','N')
    AND VOC_TOTAL_MONTH1_YN IN ('Y','N')
    AND VOC_STOP_CANCEL_MONTH1_YN IN ('Y','N')
),
segments AS (
  SELECT * FROM (VALUES
    ('STB+VOC (YYN)',  'Y','Y','N'),
    ('STB+STOP (YNY)', 'Y','N','Y'),
    ('VOC+STOP (NYY)', 'N','Y','Y')
  ) AS t(segment, stb, voc, stop)
),
agg AS (
  SELECT
    CASE
      WHEN STB_RES_1M_YN='Y' AND VOC_TOTAL_MONTH1_YN='Y' AND VOC_STOP_CANCEL_MONTH1_YN='N' THEN 'STB+VOC (YYN)'
      WHEN STB_RES_1M_YN='Y' AND VOC_TOTAL_MONTH1_YN='N' AND VOC_STOP_CANCEL_MONTH1_YN='Y' THEN 'STB+STOP (YNY)'
      WHEN STB_RES_1M_YN='N' AND VOC_TOTAL_MONTH1_YN='Y' AND VOC_STOP_CANCEL_MONTH1_YN='Y' THEN 'VOC+STOP (NYY)'
      ELSE NULL
    END AS segment,
    churn
  FROM s
),
final AS (
  SELECT
    segment,
    COUNT(*) AS cnt,
    ROUND(AVG(churn)*100, 2) AS churn_rate_percent
  FROM agg
  WHERE segment IS NOT NULL
  GROUP BY segment
)
SELECT
  seg.segment,
  COALESCE(f.cnt, 0) AS cnt,
  CASE WHEN COALESCE(f.cnt,0)=0 THEN NULL ELSE f.churn_rate_percent END AS churn_rate_percent
FROM segments seg
LEFT JOIN final f
  ON seg.segment = f.segment
ORDER BY cnt DESC;
""").fetchdf()

combo


,segment,cnt,churn_rate_percent
0,VOC+STOP (NYY),399286,12.02
1,STB+VOC (YYN),396116,9.29
2,STB+STOP (YNY),0,NaN


In [19]:
# 6) 셀: 202311 전용 3컬럼(pain_score, risk_group) 생성 + parquet 저장
SHA_202311_202312_CSV = f"../../dataset/sha_tps/sha_tps_cancel_202311_to_202312.csv"
OUT_202311_PARQUET = "result_parquet/sha_202311_pain_3col.parquet"

con.execute(f"""
DROP TABLE IF EXISTS sha_202311_pain_3col;

CREATE TABLE sha_202311_pain_3col AS
WITH src AS (
    SELECT *
    FROM read_csv(
        '{SHA_202311_202312_CSV}',
        header = true,
        union_by_name = true,
        all_varchar = true,
        strict_mode = false
    )
    WHERE p_mt = '202311'
),
mixed_key AS (
    SELECT sha2_hash, p_mt
    FROM src
    GROUP BY sha2_hash, p_mt
    HAVING COUNT(DISTINCT cancel_yn) > 1
),
filtered AS (
    SELECT s.*
    FROM src s
    LEFT JOIN mixed_key m
      ON s.sha2_hash = m.sha2_hash
     AND s.p_mt      = m.p_mt
    WHERE m.sha2_hash IS NULL
),
scored AS (
    SELECT
        sha2_hash,
        CASE WHEN COALESCE(STB_RES_1M_YN,'N')='Y' THEN 1 ELSE 0 END
      + CASE WHEN COALESCE(VOC_TOTAL_MONTH1_YN,'N')='Y' THEN 1 ELSE 0 END
      + CASE WHEN COALESCE(VOC_STOP_CANCEL_MONTH1_YN,'N')='Y' THEN 1 ELSE 0 END
        AS pain_score
    FROM filtered
)
SELECT
    sha2_hash,
    pain_score,
    CASE
        WHEN pain_score = 3 THEN '🔴 초고위험군'
        WHEN pain_score = 2 THEN '🟠 고위험군'
        WHEN pain_score = 1 THEN '🟡 위험군'
        ELSE '🟢 안정집단'
    END AS risk_group
FROM scored;
""")

# 분포 확인
con.execute("""
SELECT pain_score, COUNT(*) AS cnt
FROM sha_202311_pain_3col
GROUP BY pain_score
ORDER BY pain_score;
""").fetchdf()


,pain_score,cnt
0,0,1515833
1,1,403527
2,2,73077
3,3,6113


In [20]:
# parquet 저장
con.execute(f"""
COPY sha_202311_pain_3col
TO '{OUT_202311_PARQUET}'
(FORMAT PARQUET);
""")
print("saved:", OUT_202311_PARQUET)

saved: result_parquet/sha_202311_pain_3col.parquet


In [21]:
df_202311 = con.execute("SELECT * FROM sha_202311_pain_3col LIMIT 10").fetchdf()
df_202311


,sha2_hash,pain_score,risk_group
0,da336e1e9359d1bf19dfe1a05630776f1b4720203ac3db...,0,🟢 안정집단
1,a71d3a5b4b0aab73ea2a4c895996faaa4959b99c4ccae3...,0,🟢 안정집단
2,22c7de96dbca0e16ead4acd2387dbcab668e6bb9c7165c...,0,🟢 안정집단
3,728c7ee0ea19eb2434f558d50e6ca0f2c587b38311fce0...,0,🟢 안정집단
4,2eeebd3a667e1c5484a673a01d4344bd0f3913e36eb30e...,1,🟡 위험군
5,c928de30b7332ab71d5c71285e4e44b01b592818ceeb25...,0,🟢 안정집단
6,fd9406bde41d8fc3849f8809235b2bcfbdf0586af19a2e...,2,🟠 고위험군
7,f5268f4d5150022dbab7f5ac2f709ed30b1ea1fb2006b2...,0,🟢 안정집단
8,399ecd64015a898aa541ebf11832179e53a99c2951b83a...,0,🟢 안정집단
9,de5b534f597073f5e8850597c435a3b1c5ed6016cd42c2...,0,🟢 안정집단
